In [4]:
!cd

j:\11_Voice_Assistant\8_langGraph\AI_agent_1


**Agent Graph**

    START--> Analyze--> Router--> {calculator, greeting, time}--> END

- State is plain TypedDict(simple as possible)
- Calculator Node talks to the local MCP server(mcp_server.py)
- Greeting and Time nodes are plain local tools, no MCP needed

In [7]:
import asyncio # connecting the mcp server requires async communication
import re   # Regex  used for pattern matching in the router and for cleaning expression
from datetime import datetime   # used by the time_node to stamp the current time
from typing import Literal, Optional, TypedDict    #TypedDict keep the state as plain, simple dict-with-types

In [8]:
from langgraph.graph import END, START, StateGraph    # core graph-building scheme
from mcp import ClientSession, StdioServerParameters    #MCP Client Session+config for launching a stdio server
from mcp.client.stdio import stdio_client       #helper function that spawns the MCP server subprocess and gives us read/write streams

In [10]:
# restrict intent to exactly the three branches in the diagram(Calculator, Greeting, Time)
# using Literal instead of plain str catches typos('claculator') at type-check  time
Intent=Literal["calculator", "greeting", "time"]

In [11]:
# Our state schema
class State(TypedDict):
    # TypedDict = "just a dict" at runtime, but static tools/IDEs know the shape
    # This is intentianally the smallest state that can carry the graph
    # incoming text, the decided branch and the final answer
    query:str   #our raw input
    intent: Optional[Intent]  # filled in by the router node. None until then
    result:Optional[str]   # filled in by whichever leaf node runs; None until then